# Drive 평가 데이터에서 양자화 조건 전체 재실험

먼저 `Drive데이터_실험입력준비_Colab.ipynb`에서 만든 `drive_study_inputs_v1.json`을 업로드하세요. Colab **T4 GPU** 런타임에서 셀을 순서대로 실행합니다. Google Drive 마운트는 쓰지 않습니다. 체크포인트는 `/content/team22_drive_retest/quantized/`에 **기존 파일과 다른 이름**으로 저장됩니다. 런타임이 종료되면 체크포인트는 사라질 수 있으므로 결과 CSV/JSON 다운로드를 확인하세요.

조건은 FP16, RTN 4bit, 기존 보정 영어·한국어·64:64 혼합 GPTQ 4bit, Drive train 보정 영어·한국어·64:64 혼합 GPTQ 4bit의 **8개**입니다. 기존과 Drive의 GPTQ 설정은 `bits=4`, `group_size=128`, `sym=True`, `desc_act=False`로 같습니다. 두 보정 세트는 별도 체크포인트로 만들고, 모두 **동일한 Drive validation/test 토큰**에서 평가합니다. 기존 보정과 Drive 보정의 차이를 같은 평가 데이터 안에서 확인합니다.

Drive 보정은 연속된 train 텍스트에서 만든 512토큰 구간 128개이고 기존 보정은 긴 위키 문서 128개입니다. 보정 원천뿐 아니라 표본 구성 방식도 다르다는 한계를 기록합니다. validation은 조건 검토, test는 최종 확인용으로 보세요. test 수치로 조건을 새로 골랐다면 별도 새 표본에서 검증해야 합니다.


In [ ]:
%pip install -q "gptqmodel==7.10.0"


In [ ]:
import csv
import gc
import hashlib
import json
import math
import random
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
import gptqmodel
from google.colab import files
from IPython.display import display
from transformers import AutoModelForCausalLM
from gptqmodel import BACKEND, GPTQModel, GPTQConfig, RTNConfig

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU 런타임을 사용하세요.")

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("drive_study_inputs_v1.json 하나만 업로드하세요.")
input_bytes = next(iter(uploaded.values()))
input_sha256 = hashlib.sha256(input_bytes).hexdigest()
study = json.loads(input_bytes.decode("utf-8"))
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
if study.get("format") != "team22_drive_study_v1" or study.get("model_id") != MODEL_ID:
    raise ValueError("입력 JSON의 형식 또는 모델이 다릅니다.")
for split in ("validation", "test"):
    for lang in ("ko", "en"):
        groups = study[split][lang]
        if len(groups) != 3 or any(len(group) != 8192 for group in groups):
            raise ValueError(f"{split}/{lang}: 8192토큰 평가 묶음 3개가 필요합니다.")
for key in ("pilot_en", "pilot_ko", "drive_en", "drive_ko"):
    if len(study["calibration"][key]) != 128 or any(
        not 0 < len(ids) <= 512 for ids in study["calibration"][key]
    ):
        raise ValueError(f"보정 표본 128개가 필요합니다: {key}")

OUTPUT_DIR = Path("/content/team22_drive_retest")
MODEL_DIR = OUTPUT_DIR / "quantized"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
results = {}
print("GPU:", torch.cuda.get_device_name(0), "입력 SHA-256:", input_sha256)
print("Torch:", torch.__version__, "Transformers:", transformers.__version__,
      "GPTQModel:", gptqmodel.__version__)


In [ ]:
def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()

def evaluate(model, condition):
    model.eval()
    for split in ("validation", "test"):
        for lang in ("ko", "en"):
            for group_index, token_ids in enumerate(study[split][lang], start=1):
                total_nll = 0.0
                predicted_tokens = 0
                for start in range(0, len(token_ids), 512):
                    chunk = token_ids[start:start + 512]
                    input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
                    with torch.inference_mode():
                        output = model(input_ids=input_ids, labels=input_ids)
                    count = len(chunk) - 1
                    total_nll += output.loss.item() * count
                    predicted_tokens += count
                    del input_ids, output
                mean_nll = total_nll / predicted_tokens
                key = (condition, split, lang, group_index)
                results[key] = {
                    "condition": condition, "split": split, "language": lang,
                    "group": group_index, "mean_nll": mean_nll,
                    "ppl": math.exp(mean_nll), "predicted_tokens": predicted_tokens,
                }
    print("평가 완료:", condition)

def write_rows(name, rows):
    path = OUTPUT_DIR / name
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return path

def save_progress():
    path = write_rows("drive_quantization_progress.csv", list(results.values()))
    print("중간 결과:", path)


## 1. FP16 기준선과 RTN 4bit

모든 조건에 같은 두 언어·세 묶음을 사용합니다. 각 512토큰 구간에서 511개 예측 토큰의 평균 NLL을 모아 PPL을 계산합니다.


In [ ]:
fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0")
evaluate(fp16, "FP16")
del fp16
clear_gpu_cache()
save_progress()


In [ ]:
pool_globals = gptqmodel.get_device_thread_pool.__globals__
pool_globals["_default_cpu_workers"] = lambda: 2

def quantize_and_evaluate(condition, checkpoint_name, config, calibration_ids=None):
    path = MODEL_DIR / checkpoint_name
    weights_exist = any(path.glob("*.safetensors")) or any(path.glob("pytorch_model*.bin"))
    marker = path / "drive_input_sha256.txt"
    if (path / "config.json").exists() and weights_exist and marker.exists():
        if marker.read_text(encoding="utf-8").strip() != input_sha256:
            raise RuntimeError(f"다른 입력으로 만든 체크포인트입니다: {path}")
        print("체크포인트 재사용:", path)
    else:
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"불완전한 체크포인트를 확인하세요: {path}")
        quant_model = GPTQModel.load(
            MODEL_ID, quantize_config=config, device="cuda:0", dtype=torch.float16
        )
        calibration = ([{"input_ids": ids} for ids in calibration_ids]
                       if calibration_ids is not None else None)
        quant_model.quantize(calibration, batch_size=1, backend=BACKEND.GPTQ_TORCH)
        quant_model.save(str(path))
        marker.write_text(input_sha256, encoding="utf-8")
        del quant_model
        clear_gpu_cache()
        print("새 체크포인트:", path)
    quant_model = GPTQModel.load(
        str(path), device="cuda:0", dtype=torch.float16,
        backend=BACKEND.GPTQ_TORCH
    )
    evaluate(quant_model, condition)
    del quant_model
    clear_gpu_cache()
    save_progress()

quantize_and_evaluate(
    "RTN 4bit", "drive_eval_rtn_4bit", RTNConfig(bits=4, group_size=128)
)


## 2. 기존 보정 데이터로 새 체크포인트 생성

예비 실험의 기존 보정 표본을 재사용하되 **체크포인트와 결과는 새 이름**으로 만듭니다. 기존 데이터에서 얻은 결과와 새 평가 데이터에서 얻은 결과를 비교할 수 있습니다.


In [ ]:
def mixed_calibration(prefix):
    examples = (study["calibration"][f"{prefix}_en"][:64]
                + study["calibration"][f"{prefix}_ko"][:64])
    random.Random(22).shuffle(examples)
    assert len(examples) == 128
    return examples

for condition, name, ids in (
    ("GPTQ Pilot English 4bit", "drive_eval_gptq_pilot_en_4bit",
     study["calibration"]["pilot_en"]),
    ("GPTQ Pilot Korean 4bit", "drive_eval_gptq_pilot_ko_4bit",
     study["calibration"]["pilot_ko"]),
    ("GPTQ Pilot Mixed 64:64 4bit", "drive_eval_gptq_pilot_mixed_4bit",
     mixed_calibration("pilot")),
):
    quantize_and_evaluate(
        condition, name,
        GPTQConfig(bits=4, group_size=128, sym=True, desc_act=False), ids
    )
print("기존 보정 조건 3개 완료. 필요하면 다음 셀로 중간 CSV를 다운로드하세요.")


In [ ]:
files.download(str(OUTPUT_DIR / "drive_quantization_progress.csv"))


## 3. Drive train 보정 데이터로 별도 체크포인트 생성

영어 128구간, 한국어 128구간, 한·영 64:64 혼합을 각각 FP16 원본에서 새로 양자화합니다. 앞 단계의 모델 경로와 겹치지 않습니다.


In [ ]:
for condition, name, ids in (
    ("GPTQ Drive English 4bit", "drive_eval_gptq_drive_en_4bit",
     study["calibration"]["drive_en"]),
    ("GPTQ Drive Korean 4bit", "drive_eval_gptq_drive_ko_4bit",
     study["calibration"]["drive_ko"]),
    ("GPTQ Drive Mixed 64:64 4bit", "drive_eval_gptq_drive_mixed_4bit",
     mixed_calibration("drive")),
):
    quantize_and_evaluate(
        condition, name,
        GPTQConfig(bits=4, group_size=128, sym=True, desc_act=False), ids
    )


## 4. 결과 정리와 브라우저 다운로드

각 split·언어·묶음의 **같은 FP16**을 분모로 `D_percent = (양자화 PPL / FP16 PPL − 1) × 100`을 계산합니다. 한국어 PPL만 보면 한국어 보정 조건을, 영어는 부작용을 확인하는 지표로 봅니다. validation에서 후보를 검토한 뒤 test를 최종 결과로 사용하세요.


In [ ]:
conditions = {
    "FP16", "RTN 4bit",
    "GPTQ Pilot English 4bit", "GPTQ Pilot Korean 4bit",
    "GPTQ Pilot Mixed 64:64 4bit", "GPTQ Drive English 4bit",
    "GPTQ Drive Korean 4bit", "GPTQ Drive Mixed 64:64 4bit",
}
if {key[0] for key in results} != conditions or len(results) != 96:
    raise RuntimeError(f"8조건 × 2분할 × 2언어 × 3묶음 결과 96행이 필요합니다: {len(results)}행")
base = {(split, lang, group): results["FP16", split, lang, group]["ppl"]
        for split in ("validation", "test") for lang in ("ko", "en")
        for group in (1, 2, 3)}
all_rows = []
for row in results.values():
    row = row.copy()
    row["D_percent"] = (
        row["ppl"] / base[row["split"], row["language"], row["group"]] - 1
    ) * 100
    all_rows.append(row)
all_rows.sort(key=lambda r: (r["split"], r["condition"], r["language"], r["group"]))
summary = []
for split in ("validation", "test"):
    for condition in sorted(conditions):
        for lang in ("ko", "en"):
            group_rows = [row for row in all_rows if row["split"] == split
                          and row["condition"] == condition
                          and row["language"] == lang]
            summary.append({
                "split": split, "condition": condition, "language": lang,
                "mean_ppl": sum(row["ppl"] for row in group_rows) / 3,
                "mean_D_percent": sum(row["D_percent"] for row in group_rows) / 3,
            })
raw_path = write_rows("drive_quantization_ppl_by_group.csv", all_rows)
summary_path = write_rows("drive_quantization_summary.csv", summary)
run_info = {
    "model_id": MODEL_ID, "input_sha256": input_sha256,
    "source_sha256": study["source_sha256"],
    "torch": torch.__version__, "transformers": transformers.__version__,
    "gptqmodel": gptqmodel.__version__, "gpu": torch.cuda.get_device_name(0),
    "gptq": {"bits": 4, "group_size": 128, "sym": True, "desc_act": False},
    "checkpoint_dir": str(MODEL_DIR),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}
info_path = OUTPUT_DIR / "drive_quantization_runtime.json"
info_path.write_text(json.dumps(run_info,ensure_ascii=False,indent=2),encoding="utf-8")
print("묶음별 결과 96행, 언어별 요약 32행")
display(summary)


In [ ]:
files.download(str(raw_path))


In [ ]:
files.download(str(summary_path))


In [ ]:
files.download(str(info_path))
